# Multilingual Document OCR + MRZ Parser
### Domino Data Lab · Arabic / French / English · Fully Offline · v7

---

## All bugs fixed

| Version | Error | Fix |
|---------|-------|-----|
| v1 | `AssertionError: lang must in dict_keys, got ar,fr,en` | Two engines: `lang='ar'` + `lang='fr'` |
| v2–v3 | `ConnectionError` despite `det_model_dir` | Copy models into `~/.paddleocr/whl/` cache |
| v4 | Models in wrong folder | Read `BASE_DIR` from PaddleOCR source directly |
| v5 | Still downloading | `enable_hpi=True` needs `.onnx` files → `enable_hpi=False` |
| **v7** | `Cannot read: carte idd.PDF` | **PDF support added via `pymupdf` — each page → image → OCR** |

## How PDF processing works
1. `pymupdf` opens the PDF and renders each page at **300 DPI** to a numpy array.
2. Each page is treated as a separate image and passed through both OCR engines.
3. Results from all pages are merged into one result dict per PDF file.
4. MRZ is searched across all pages combined.

---
## Cell 0 — Install pymupdf (run once)
Only needed the first time. Safe to skip on subsequent runs.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'pymupdf==1.24.1', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '')
print(result.stderr[-300:] if result.stderr else '')

import fitz  # pymupdf
print(f'✓ Cell 0 — pymupdf {fitz.version[0]} ready')

---
## Cell 1 — Imports

In [ ]:
import json
import logging
import os
import re
import shutil
import time
from pathlib import Path
from typing import Any

import cv2
import fitz                    # pymupdf — PDF → image
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

print('✓ Cell 1 — imports OK')

---
## Cell 2 — Configuration, Model Setup, Download Blocker

In [ ]:
import requests as _requests

# =============================================================================
# ★ EDIT THESE IF NEEDED ★
# =============================================================================
INPUT_DIR  = Path('/mnt/ocr_data/documents')   # folder containing your PDF/image files
MODELS_DIR = Path('/mnt/ocr_data/models')      # folder containing the 3 extracted model folders

# =============================================================================
# OUTPUT PATHS
# =============================================================================
OUTPUT_JSON  = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL = Path('/mnt/artifacts/report.xlsx')

# =============================================================================
# PERFORMANCE
# =============================================================================
MAX_DIMENSION = 1280        # px — resize images larger than this before OCR
PDF_DPI       = 300         # DPI used when rendering PDF pages to images
                            # 300 = good quality, increase to 400 if text is tiny
CPU_THREADS   = os.cpu_count() or 4

# =============================================================================
# SUPPORTED FORMATS — now includes PDF
# =============================================================================
SUPPORTED_IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
SUPPORTED_PDF_EXT   = {'.pdf'}
SUPPORTED_ALL_EXT   = SUPPORTED_IMAGE_EXT | SUPPORTED_PDF_EXT

MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

# =============================================================================
# FIND PADDLEOCR BASE_DIR FROM ITS OWN SOURCE
# =============================================================================
try:
    from paddleocr.paddleocr import BASE_DIR as _PADDLE_BASE
    PADDLE_BASE = Path(_PADDLE_BASE)
    print(f'✓ PaddleOCR BASE_DIR : {PADDLE_BASE}')
except Exception as exc:
    import paddleocr as _poc
    PADDLE_BASE = Path(_poc.__file__).parent.parent.parent.parent / '.paddleocr'
    print(f'⚠ BASE_DIR fallback  : {PADDLE_BASE}  ({exc})')

# Cache paths
CACHE_DET   = PADDLE_BASE / 'whl' / 'det' / 'multilingual' / 'Multilingual_PP-OCRv3_det_infer'
CACHE_AR    = PADDLE_BASE / 'whl' / 'rec' / 'arabic'        / 'arabic_PP-OCRv3_rec_infer'
CACHE_LATIN = PADDLE_BASE / 'whl' / 'rec' / 'latin'         / 'latin_PP-OCRv3_rec_infer'

# Source paths
SRC_DET     = MODELS_DIR / 'Multilingual_PP-OCRv3_det_infer'
SRC_AR      = MODELS_DIR / 'arabic_PP-OCRv3_rec_infer'
SRC_LATIN   = MODELS_DIR / 'latin_PP-OCRv3_rec_infer'

# =============================================================================
# COPY MODELS INTO CACHE
# =============================================================================
def _is_valid_model_dir(p: Path) -> bool:
    return p.is_dir() and (p / 'inference.pdmodel').exists()

def ensure_cached(src: Path, dst: Path, label: str) -> None:
    if _is_valid_model_dir(dst):
        print(f'  ✓ {label:35s} already in cache')
        return
    if not _is_valid_model_dir(src):
        raise FileNotFoundError(
            f'\n❌ Model missing at: {src}\n'
            f'  Upload to {MODELS_DIR}/ via Domino browser.\n'
            f'  Needs: inference.pdmodel  inference.pdiparams  inference.pdiparams.info\n'
        )
    print(f'  → Copying {label} …')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists(): shutil.rmtree(str(dst))
    shutil.copytree(str(src), str(dst))
    assert _is_valid_model_dir(dst), f'Copy failed at {dst}'
    print(f'  ✓ {label:35s} copied OK')

print('\nEnsuring models are in cache …')
ensure_cached(SRC_DET,   CACHE_DET,   'Detection model')
ensure_cached(SRC_AR,    CACHE_AR,    'Arabic rec model')
ensure_cached(SRC_LATIN, CACHE_LATIN, 'Latin rec model (fr/en)')

# =============================================================================
# BLOCK ALL DOWNLOADS
# =============================================================================
_orig_send = _requests.Session.send

def _no_paddle_downloads(self, request, **kwargs):
    url = getattr(request, 'url', str(request))
    if 'bcebos.com' in url:
        raise RuntimeError(
            f'\n\n❌ PaddleOCR tried to download: {url}\n'
            f'   A model file is missing from cache.\n'
            f'   Check: ls {CACHE_DET}/  |  ls {CACHE_AR}/  |  ls {CACHE_LATIN}/\n'
        )
    return _orig_send(self, request, **kwargs)

_requests.Session.send = _no_paddle_downloads

print('\n✓ Download blocker active')
print(f'✓ Cell 2 complete  |  threads={CPU_THREADS}  |  input={INPUT_DIR}')

# Warn if input dir missing
if not INPUT_DIR.exists():
    print(f'\n  ⚠ WARNING: {INPUT_DIR} not found — check INPUT_DIR path above')
else:
    files = [p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in SUPPORTED_ALL_EXT]
    print(f'  Found {len(files)} supported file(s) in {INPUT_DIR}')
    for f in files:
        print(f'    {f.name}')

---
## Cell 3 — Image and PDF Loading

**New in v7:** `pdf_to_images()` converts each PDF page to a BGR numpy array  
using `pymupdf` at 300 DPI. No poppler, no ghostscript, no external tools needed.

In [ ]:
def load_and_resize(img: np.ndarray, max_dim: int = MAX_DIMENSION) -> np.ndarray:
    """
    Downscale a BGR numpy array if its longest side exceeds max_dim.
    cv2.INTER_AREA gives best quality when shrinking.
    """
    h, w  = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)
    if scale < 1.0:
        img = cv2.resize(img, (int(w * scale), int(h * scale)),
                         interpolation=cv2.INTER_AREA)
    return img


def load_image_file(image_path: Path, max_dim: int = MAX_DIMENSION) -> np.ndarray:
    """
    Load a standard image file (jpg, png, etc.) with cv2 and resize it.

    Raises FileNotFoundError if cv2 cannot open the file.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(
            f'cv2 cannot read: {image_path}\n'
            f'Make sure it is a real image file (jpg/png/bmp/tiff/webp).\n'
            f'PDFs must go through pdf_to_images() instead.'
        )
    return load_and_resize(img, max_dim)


def pdf_to_images(
    pdf_path: Path,
    dpi:      int = PDF_DPI,
    max_dim:  int = MAX_DIMENSION,
) -> list[np.ndarray]:
    """
    Render every page of a PDF into a BGR numpy array.

    Uses pymupdf (fitz) — no poppler or ghostscript needed.
    Each page is rendered at `dpi` dots per inch then resized
    to max_dim on its longest side for OCR efficiency.

    Parameters
    ----------
    pdf_path : Path   Path to the .pdf file.
    dpi      : int    Render resolution. 300 is good for most scans.
                      Use 400 if the text is very small.
    max_dim  : int    Resize threshold after rendering.

    Returns
    -------
    list[np.ndarray]   One BGR array per page.
    """
    doc    = fitz.open(str(pdf_path))
    pages  = []
    matrix = fitz.Matrix(dpi / 72, dpi / 72)   # 72 = PDF default DPI

    for page_num in range(len(doc)):
        page   = doc[page_num]
        pixmap = page.get_pixmap(matrix=matrix, colorspace=fitz.csRGB)

        # Convert pixmap → numpy array (RGB) → BGR for OpenCV / PaddleOCR
        img_rgb = np.frombuffer(pixmap.samples, dtype=np.uint8).reshape(
            pixmap.height, pixmap.width, 3
        )
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        img_bgr = load_and_resize(img_bgr, max_dim)

        pages.append(img_bgr)
        log.debug('  PDF page %d/%d  size=%dx%d',
                  page_num + 1, len(doc), img_bgr.shape[1], img_bgr.shape[0])

    doc.close()
    return pages


print('✓ Cell 3 — load_image_file and pdf_to_images defined')
print(f'  PDF render DPI  : {PDF_DPI}')
print(f'  Max image dim   : {MAX_DIMENSION} px after rendering')

---
## Cell 4 — Build the Two OCR Engines (offline)

In [ ]:
from paddleocr import PaddleOCR


def _make_engine(lang: str, rec_model_dir: Path) -> PaddleOCR:
    """
    Create a single-language PaddleOCR engine — fully offline.

    enable_hpi=False  : CRITICAL — True would need .onnx files which we don't have,
                        causing a download attempt even with models in cache.
    det_model_dir     : explicit path — double safety net.
    rec_model_dir     : explicit path — double safety net.
    use_angle_cls=False : disables cls model entirely (no third download).
    """
    return PaddleOCR(
        lang=lang,
        det_model_dir=str(CACHE_DET),
        rec_model_dir=str(rec_model_dir),
        use_angle_cls=False,
        use_gpu=False,
        enable_hpi=False,           # ← KEY: False avoids .onnx download
        cpu_threads=CPU_THREADS,
        show_log=False,
    )


print('Initialising Arabic engine  (lang="ar") …')
ocr_ar = _make_engine('ar', CACHE_AR)
print('✓ Arabic engine ready')

print()
print('Initialising Latin engine   (lang="fr") …')
ocr_latin = _make_engine('fr', CACHE_LATIN)
print('✓ Latin engine ready  (French + English)')

print()
print('✓ Cell 4 complete — both engines ready, fully offline')

---
## Cell 5 — MRZ Detection and Parsing

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """Scan OCR lines for ICAO MRZ candidates (A-Z, 0-9, '<', length >= 30)."""
    mrz = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz.append(cleaned)
    return mrz


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """Parse + validate MRZ using OmniMRZ (ICAO 9303)."""
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed — MRZ parsing skipped.')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}
    try:
        obj = MRZ('\n'.join(mrz_lines))
        fields = {k: getattr(obj, k, None) for k in [
            'document_type', 'issuing_country', 'document_number',
            'surname', 'given_names', 'nationality',
            'birth_date', 'sex', 'expiry_date',
            'optional_data_1', 'optional_data_2',
        ]}
        validation: dict[str, Any] = {}
        for attr in dir(obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try: validation[attr] = getattr(obj, attr)
                except Exception: pass
        overall = getattr(obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall
        return {'raw_mrz': mrz_lines, 'parsed_fields': fields, 'validation': validation}
    except Exception as exc:
        log.warning('OmniMRZ failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('✓ Cell 5 — MRZ helpers defined')

---
## Cell 6 — Document Processor (images + PDFs)

**New in v7:** `process_document()` detects whether the file is a PDF or an image.
- **PDF** → renders all pages with `pdf_to_images()` → OCR each page → merge all text
- **Image** → `load_image_file()` → OCR → merge text

Both cases search the merged text for MRZ.

In [ ]:
def _extract_lines(engine: Any, img: np.ndarray) -> list[str]:
    """Run one PaddleOCR engine on one image array, return list of text strings."""
    lines: list[str] = []
    raw = engine.ocr(img, cls=False)
    if raw and raw[0]:
        for item in raw[0]:
            if item and len(item) >= 2:
                txt = item[1][0] if isinstance(item[1], (list, tuple)) else str(item[1])
                lines.append(txt)
    return lines


def _ocr_one_image(
    img:          np.ndarray,
    engine_ar:    Any,
    engine_latin: Any,
) -> list[str]:
    """
    Run both engines on one image, merge results, remove duplicates.
    Arabic lines come first.
    """
    lines_ar    = _extract_lines(engine_ar,    img)
    lines_latin = _extract_lines(engine_latin, img)
    seen: set[str]        = set()
    merged: list[str]     = []
    for line in lines_ar + lines_latin:
        if line not in seen:
            seen.add(line)
            merged.append(line)
    return merged


def process_document(
    file_path:    str | Path,
    engine_ar:    Any,
    engine_latin: Any,
    max_dim:      int = MAX_DIMENSION,
    pdf_dpi:      int = PDF_DPI,
) -> dict[str, Any]:
    """
    Run OCR on one document — works for BOTH image files AND PDF files.

    For PDFs: renders every page then OCRs each one.
              Text from all pages is merged into full_text.
              MRZ is searched across all pages combined.

    Returns dict:
        file_name   — filename
        file_type   — 'pdf' or 'image'
        page_count  — number of pages (1 for images)
        full_text   — all OCR text, newline-separated
        has_mrz     — True if MRZ was found
        mrz_data    — parsed MRZ or None
        error       — error message or None
        elapsed_sec — processing time
    """
    file_path = Path(file_path)
    result: dict[str, Any] = {
        'file_name':   file_path.name,
        'file_type':   'pdf' if file_path.suffix.lower() == '.pdf' else 'image',
        'page_count':  0,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()

    try:
        is_pdf = file_path.suffix.lower() == '.pdf'

        if is_pdf:
            # ── PDF: render all pages then OCR each one ───────────────────
            log.info('  PDF detected — rendering pages at %d DPI …', pdf_dpi)
            page_images = pdf_to_images(file_path, dpi=pdf_dpi, max_dim=max_dim)
            result['page_count'] = len(page_images)
            log.info('  Rendered %d page(s)', len(page_images))

            all_lines: list[str] = []
            for i, page_img in enumerate(page_images):
                log.info('  OCR page %d/%d …', i + 1, len(page_images))
                page_lines = _ocr_one_image(page_img, engine_ar, engine_latin)
                all_lines.extend(page_lines)
                log.info('    → %d line(s) on page %d', len(page_lines), i + 1)

            # Deduplicate across pages while preserving order
            seen: set[str]       = set()
            text_lines: list[str] = []
            for line in all_lines:
                if line not in seen:
                    seen.add(line)
                    text_lines.append(line)

        else:
            # ── Image: load directly ──────────────────────────────────────
            img  = load_image_file(file_path, max_dim)
            result['page_count'] = 1
            text_lines = _ocr_one_image(img, engine_ar, engine_latin)

        result['full_text'] = '\n'.join(text_lines)
        log.info('  Total text lines: %d', len(text_lines))

        # ── MRZ detection across all pages ────────────────────────────────
        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected (%d lines)', len(mrz_lines))
        else:
            log.info('  — No MRZ found')

    except Exception as exc:
        log.error('Error %s: %s', file_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('✓ Cell 6 — process_document defined (PDF + image support)')

---
## Cell 7 — Test with Your File
Change `SAMPLE_FILE` to the path of your `carte idd.PDF` or any image.

In [ ]:
# ── Change this to your actual file ──────────────────────────────────────────
SAMPLE_FILE = Path('/mnt/ocr_data/documents/carte idd.PDF')  # ← your file

if SAMPLE_FILE.exists():
    suffix = SAMPLE_FILE.suffix.lower()
    print(f'Testing: {SAMPLE_FILE.name}  (type: {suffix})')

    r = process_document(SAMPLE_FILE, ocr_ar, ocr_latin)

    print(f'\n  File       : {r["file_name"]}')
    print(f'  Type       : {r["file_type"]}')
    print(f'  Pages      : {r["page_count"]}')
    print(f'  Has MRZ    : {r["has_mrz"]}')
    print(f'  Time       : {r["elapsed_sec"]}s')
    print(f'  Error      : {r["error"]}')
    print(f'\n--- OCR TEXT ---')
    print(r['full_text'] or '(nothing detected)')

    if r['has_mrz']:
        print('\n--- MRZ ---')
        print(json.dumps(r['mrz_data'], ensure_ascii=False, indent=2, default=str))

    print('\n✓ Cell 7 complete')
else:
    print(f'[SKIP] {SAMPLE_FILE} not found.')
    print('Change SAMPLE_FILE path above to your actual file.')

---
## Cell 8 — Batch Processing (all files)

In [ ]:
def collect_files(input_path: Path) -> list[Path]:
    """
    Return sorted list of supported files from a file or directory (recursive).
    Supports both image files AND PDF files.
    """
    if input_path.is_file():
        return [input_path]
    files = sorted(
        p for p in input_path.rglob('*')
        if p.suffix.lower() in SUPPORTED_ALL_EXT
    )
    if not files:
        print(f'⚠ No supported files found in {input_path}')
        print(f'  Supported: {sorted(SUPPORTED_ALL_EXT)}')
    return files


files = collect_files(INPUT_DIR)
pdfs   = [f for f in files if f.suffix.lower() == '.pdf']
images = [f for f in files if f.suffix.lower() != '.pdf']
print(f'Found {len(files)} file(s) — {len(pdfs)} PDF(s), {len(images)} image(s):')
for f in files:
    print(f'  {f.name}')

print()
all_results: list[dict[str, Any]] = []

for file_path in tqdm(files, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', file_path.name)
    r = process_document(file_path, ocr_ar, ocr_latin)
    all_results.append(r)
    log.info('  %.2fs | pages=%s | has_mrz=%s | error=%s',
             r['elapsed_sec'], r['page_count'], r['has_mrz'], r['error'])

total = sum(r['elapsed_sec'] for r in all_results)
print(f'\n✓ Cell 8 — {len(all_results)} file(s) done in {total:.1f}s')

---
## Cell 9 — Save JSON

In [ ]:
def save_json(results: list[dict[str, Any]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    print(f'  {out.stat().st_size/1024:.1f} KB written')


save_json(all_results, OUTPUT_JSON)
print(f'✓ Cell 9 — JSON → {OUTPUT_JSON}')

---
## Cell 10 — Save Excel (Summary · MRZ Details · Full Text)

In [ ]:
def _style_ws(ws, max_w: int = 60) -> None:
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best   = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_w)


def build_excel(results: list[dict[str, Any]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    s_rows, m_rows, t_rows = [], [], []

    for r in results:
        md  = r.get('mrz_data') or {}
        val = md.get('validation', {}) if r.get('has_mrz') else {}
        ov  = val.get('overall_valid')

        s_rows.append({
            'file_name': r.get('file_name'), 'file_type': r.get('file_type'),
            'page_count': r.get('page_count'), 'has_mrz': r.get('has_mrz'),
            'overall_mrz_valid': ov, 'elapsed_sec': r.get('elapsed_sec'),
            'error': r.get('error'),
        })
        t_rows.append({'file_name': r.get('file_name'), 'full_text': r.get('full_text')})

        if r.get('has_mrz') and md:
            pf = md.get('parsed_fields', {}) or {}
            m_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(md.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'overall_valid':   ov,
                **{f'valid_{k}': v for k, v in val.items() if k != 'overall_valid'},
            })

    with pd.ExcelWriter(str(out), engine='openpyxl') as w:
        pd.DataFrame(s_rows).to_excel(w, sheet_name='Summary',     index=False)
        (pd.DataFrame(m_rows) if m_rows else pd.DataFrame(columns=['file_name'])
         ).to_excel(w, sheet_name='MRZ Details', index=False)
        pd.DataFrame(t_rows).to_excel(w, sheet_name='Full Text',   index=False)

    wb = load_workbook(str(out))
    for ws in wb.worksheets:
        _style_ws(ws)
    wb.save(str(out))
    print(f'  {out.stat().st_size/1024:.1f} KB written')


build_excel(all_results, OUTPUT_EXCEL)
print(f'✓ Cell 10 — Excel → {OUTPUT_EXCEL}')

---
## Cell 11 — Final Summary

In [ ]:
df      = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')
n       = len(df)
total_t = sum(r['elapsed_sec'] for r in all_results)

print('=' * 52)
print('  RESULTS SUMMARY')
print('=' * 52)
print(f'  Files processed    : {n}')
print(f'  PDFs               : {len([r for r in all_results if r["file_type"]=="pdf"])}')
print(f'  Images             : {len([r for r in all_results if r["file_type"]=="image"])}')
print(f'  With MRZ           : {int(df["has_mrz"].sum())}')
print(f'  MRZ valid          : {int(df["overall_mrz_valid"].sum())}')
print(f'  Errors             : {int(df["error"].notna().sum())}')
print(f'  Total time         : {total_t:.1f}s')
print(f'  Avg per file       : {total_t/max(n,1):.2f}s')
print('=' * 52)
print(f'  JSON  → {OUTPUT_JSON}')
print(f'  Excel → {OUTPUT_EXCEL}')
print('=' * 52)
print('Download: Domino → Jobs → Artifacts')
df